In [1]:
import numpy as np
import pandas as pd
import os
import pathlib
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import yaml
import pathlib
from pathlib import Path
import shutil
import io
import re

In [2]:
#--------------------------------------------------------------------------------------------------------------------------
# read path
#--------------------------------------------------------------------------------------------------------------------------

before_dir = Path("Artemide/unpolDY")

#--------------------------------------------------------------------------------------------------------------------------
# write path
#--------------------------------------------------------------------------------------------------------------------------

after_dir = Path("Raw")

Auxilary Functions

In [3]:
def read_csv(path):

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()
    hdr = next(i for i, ln in enumerate(lines) if ln.strip().startswith("Point id"))

    meta_df = pd.read_csv(
        io.StringIO("".join(lines[:hdr])),
        names=["key","value"],
        usecols=[0,1],
        engine="python",
        on_bad_lines="skip",
    )
    meta_df = meta_df.dropna(subset=["value"]) 
    meta = dict(zip(meta_df["key"].str.strip(), meta_df["value"].astype(str).str.strip()))

    df = pd.read_csv(io.StringIO("".join(lines[hdr:])), engine="python")

    return meta, df

DY

In [4]:
processes = ["DY","SIDIS"]

for process in processes:
    process_dir = after_dir / process
    if process_dir.exists():
        shutil.rmtree(process_dir) 
    process_dir.mkdir(parents=True, exist_ok=True)

Test

In [5]:
experiment = "ATLAS_7"
specific_name = None

In [6]:
exp_dir = before_dir / experiment

if specific_name == None:
    file_name = [file.stem for file in sorted(exp_dir.glob("*.csv"))][0]
else:
    file_name = specific_name

path = exp_dir / f"{file_name}.csv"

meta, df_raw = read_csv(path)

def get_scalar(name):
    val = meta[name]
    if val.lower() == "true":
        return True
    elif val.lower() == "false":
        return False
    else:
        return val

def get_array(name):
    return np.array(df_raw[name])

# arrays

sqrts = np.sqrt(get_array("s[GeV^2]"))
print('sqrts =',sqrts)

qT_min = get_array("qTMin[GeV]")
qT_max = get_array("qTMax[GeV]")
qT_mean = get_array("<qT>[GeV]")
print('qT_min, qT_max, qT_mean =', qT_min, qT_max, qT_mean)

Q_min = get_array("Qmin[GeV]")
Q_max = get_array("Qmax[GeV]")
Q_mean = get_array("<Q>[GeV]")
print('Q_min, Q_max, Q_mean =', Q_min, Q_max, Q_mean)

y_min = get_array("yMin")
y_max = get_array("yMax")
y_mean = get_array("<y>")
print('y_min, y_max, y_mean =', y_min, y_max, y_mean)

xsec_array = get_array("xSec")
print('xsec_array =', xsec_array)

# fiducial cuts
If_fiducial_cut = get_array("FiducialCuts")
pT_min_1 = get_array("kCut1[GeV]")
pT_min_2 = get_array("kCut2[GeV]")
eta_min = get_array("etaMin")
eta_max = get_array("etaMax")
print('If_fiducial_cut, pT_min_1, pT_min_2, eta_min, eta_max =', If_fiducial_cut, pT_min_1, pT_min_2, eta_min, eta_max)

# normalization
norm_error = get_scalar("List of norm.errors (relative)")
if_normalized = get_scalar("Total cross-section nomalized")
print("norm_error, if_normalized =", norm_error, if_normalized)

def count_errors(label):
    pat = re.compile(rf"^{re.escape(label)}\d+$")
    return sum(1 for c in df_raw.columns if pat.match(c))
count_errors('Uncorr.Err.')

def get_errors(label, i):
    return get_array(f"{label}{i}")

def push_errors(df, label, name):
    for i in range(count_errors(label)):
        df[f"label{i}"] = get_errors(label, i)
    return df   

print("N_correlated_errors =", count_errors('Corr.Err.'))
print("N_uncorrelated_errors =", count_errors('Uncorr.Err.'))
#df = push_errors(df, 'Uncorr.Err.', 'unc')
#df = push_errors(df, 'Corr.Err.', 'add')

sqrts = [7000. 7000. 7000. 7000. 7000. 7000. 7000. 7000. 7000. 7000. 7000. 7000.
 7000. 7000. 7000. 7000. 7000. 7000. 7000. 7000. 7000. 7000. 7000. 7000.
 7000. 7000.]
qT_min, qT_max, qT_mean = [  0.   2.   4.   6.   8.  10.  12.  14.  16.  18.  22.  26.  30.  34.
  38.  42.  46.  50.  54.  60.  70.  80. 100. 150. 200. 300.] [  2.   4.   6.   8.  10.  12.  14.  16.  18.  22.  26.  30.  34.  38.
  42.  46.  50.  54.  60.  70.  80. 100. 150. 200. 300. 800.] [  1.   3.   5.   7.   9.  11.  13.  15.  17.  20.  24.  28.  32.  36.
  40.  44.  48.  52.  57.  65.  75.  90. 125. 175. 250. 550.]
Q_min, Q_max, Q_mean = [66. 66. 66. 66. 66. 66. 66. 66. 66. 66. 66. 66. 66. 66. 66. 66. 66. 66.
 66. 66. 66. 66. 66. 66. 66. 66.] [116. 116. 116. 116. 116. 116. 116. 116. 116. 116. 116. 116. 116. 116.
 116. 116. 116. 116. 116. 116. 116. 116. 116. 116. 116. 116.] [91. 91. 91. 91. 91. 91. 91. 91. 91. 91. 91. 91. 91. 91. 91. 91. 91. 91.
 91. 91. 91. 91. 91. 91. 91. 91.]
y_min, y_max, y_mean = [0. 0. 0. 0. 0

Isoscalarities

In [7]:
isoscalarity_dict = dict([])

isoscalarity_dict["ATLAS_7"]  = 1
isoscalarity_dict["ATLAS_8"]  = 1
isoscalarity_dict["ATLAS_13"] = 1
isoscalarity_dict["CDF_I"]    = -1
isoscalarity_dict["CDF_II"]   = -1
isoscalarity_dict["CMS_7"]    = 1
isoscalarity_dict["CMS_8"]    = 1
isoscalarity_dict["CMS_13"]   = 1
isoscalarity_dict["D0_I"]     = -1
isoscalarity_dict["D0_II"]    = -1
isoscalarity_dict["D0_II_mu"] = -1
isoscalarity_dict["E288"]     = 0.4603
isoscalarity_dict["E605"]     = 0.4603
isoscalarity_dict["E772"]     = 0.5
isoscalarity_dict["LHCb_7"]   = 1
isoscalarity_dict["LHCb_8"]   = 1
isoscalarity_dict["LHCb_13"]  = 1
isoscalarity_dict["PHENIX"]   = 1
isoscalarity_dict["STAR"]     = 1

Observable

In [8]:
observable_dict = dict([])

observable_dict["ATLAS_7"]  = "1/σ*dσ/dqT"
observable_dict["ATLAS_8"]  = "dσ/dqT"
observable_dict["ATLAS_13"] = "1/σ*dσ/dqT"
observable_dict["CDF_I"]    = "dσ/dqT"
observable_dict["CDF_II"]   = "dσ/dqT"
observable_dict["CMS_7"]    = "1/σ*dσ/dqT"
observable_dict["CMS_8"]    = "1/σ*dσ/dqT"
observable_dict["CMS_13"]   = "dσ/dqT"
observable_dict["D0_I"]     = "dσ/dqT"
observable_dict["D0_II"]    = "1/σ*dσ/dqT"
observable_dict["D0_II_mu"] = "1/σ*dσ/dqT"
observable_dict["E288"]     = "E*dσ/d3q"
observable_dict["E605"]     = "E*dσ/d3q(xf)"
observable_dict["E772"]     = "E*dσ/d3q(xf)"
observable_dict["LHCb_7"]   = "dσ/dqT"
observable_dict["LHCb_8"]   = "dσ/dqT"
observable_dict["LHCb_13"]  = "dσ/dqT"
observable_dict["PHENIX"]   = "dσ/dqT"
observable_dict["STAR"]     = "dσ/dqT"

y dependent

In [9]:
experiments =[
    'ATLAS_7',
    'ATLAS_8',
    'ATLAS_13',  
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    'PHENIX',
    'STAR'
]
excludes = ["CMS13-50Q76","CMS13-76Q106"]#["CMS13-50Q76","CMS13-76Q106","CMS13-106Q170","CMS13-170Q350","CMS13-350Q1000"]

qT_if_integrate = True
Q_if_integrate = True
y_if_integrate = True

In [10]:
negative_y_files = []

for experiment in experiments:

    exp_dir = before_dir/experiment
    file_names = [file.stem for file in sorted(exp_dir.glob("*.csv"))]

    save_dir = Path(after_dir) / "DY" / experiment
    save_dir.mkdir(parents=True, exist_ok=True)

    for file_name in (f for f in file_names if f not in excludes):

        path = exp_dir / f"{file_name}.csv"

        meta, df_raw = read_csv(path)

        def get_scalar(name):
            val = meta[name]
            if val.lower() == "true":
                return True
            elif val.lower() == "false":
                return False
            else:
                return val

        def get_array(name):
            return np.array(df_raw[name])

        # safety

        observable = observable_dict[experiment]
        if_normalized = get_scalar("Total cross-section nomalized")
        if if_normalized==True and observable!="1/σ*dσ/dqT":
            raise ValueError("observable type wrong")
        if if_normalized==False and observable=="1/σ*dσ/dqT":
            raise ValueError("observable type wrong")        
        
        isoscalarity = isoscalarity_dict[experiment]
        y_min = get_array("yMin")[0]
        y_max = get_array("yMax")[0]
        if isoscalarity == 1 and y_min < 0:   
            negative_y_files.append(file_name)
            if abs(y_min) != y_max:
                raise ValueError("non-symmetric y bin for pp")

        # arrays

        sqrts_array = np.sqrt(get_array("s[GeV^2]"))
        isoscalarity = isoscalarity_dict[experiment]

        qT_min_array = get_array("qTMin[GeV]")
        qT_max_array = get_array("qTMax[GeV]")
        qT_mean_array = get_array("<qT>[GeV]")

        Q_min_array = get_array("Qmin[GeV]")
        Q_max_array = get_array("Qmax[GeV]")
        Q_mean_array = get_array("<Q>[GeV]")

        y_min_array = get_array("yMin")
        y_max_array = get_array("yMax")
        y_mean_array = get_array("<y>")

        xsec_array = get_array("xSec")

        # fiducial cuts
        if_fiducial = get_array("FiducialCuts")
        pT_min_1 = get_array("kCut1[GeV]")
        pT_min_2 = get_array("kCut2[GeV]")
        eta_min = get_array("etaMin")
        eta_max = get_array("etaMax")

        # normalization
        norm_error = get_scalar("List of norm.errors (relative)")
        if_normalized = get_scalar("Total cross-section nomalized")

        # generate csv

        N = len(xsec_array)

        df = pd.DataFrame({

            "qT_if_integrate": [qT_if_integrate]*N,
            "qT_mean": qT_mean_array, "qT_min": qT_min_array, "qT_max": qT_max_array,

            "Q_if_integrate": [Q_if_integrate]*N,
            "Q_mean": Q_mean_array, "Q_min": Q_min_array, "Q_max": Q_max_array,

            "y_if_integrate": [y_if_integrate]*N,
            "y_mean": y_mean_array, "y_min": y_min_array, "y_max": y_max_array,

            "sqrts": sqrts_array,
            "isoscalarity": [isoscalarity]*N,
            "observable": [observable]*N,

            "if_fiducial": if_fiducial,
            "pT_min_1": pT_min_1,
            "pT_min_2": pT_min_2,
            "eta_min": eta_min,
            "eta_max": eta_max
        })

        # push errors

        def count_errors(label):
            pat = re.compile(rf"^{re.escape(label)}\d+$")
            return sum(1 for c in df_raw.columns if pat.match(c))
        count_errors('Uncorr.Err.')

        def get_errors(label, i):
            return get_array(f"{label}{i}")

        def push_errors(df, label, name):
            for i in range(count_errors(label)):
                df[f"{name}{i}"] = get_errors(label, i)
            return df   

        df["xsec"] = xsec_array

        df = push_errors(df, 'Uncorr.Err.', 'error_unc_')
        df = push_errors(df, 'Corr.Err.', 'error_add_')
        df["error_mult_0 "] = [norm_error]*N

        destination = after_dir / "DY" / experiment / f"{file_name}.csv"
        if destination.exists():
            raise FileExistsError(f"{file_name} already exists")
        df_rounded = df.round(6)
        df_rounded.to_csv(destination, index=False)

        print(f"{file_name} generated")

ATLAS7-00y10 generated
ATLAS7-10y20 generated
ATLAS7-20y24 generated
ATLAS8-00y04 generated
ATLAS8-04y08 generated
ATLAS8-08y12 generated
ATLAS8-116Q150 generated
ATLAS8-12y16 generated
ATLAS8-16y20 generated
ATLAS8-20y24 generated
ATLAS8-46Q66 generated
ATLAS13 generated
CDF1 generated
CDF2 generated
CMS7 generated
CMS8 generated
CMS13-00y04 generated
CMS13-04y08 generated
CMS13-08y12 generated
CMS13-106Q170 generated
CMS13-12y16 generated
CMS13-16y24 generated
CMS13-170Q350 generated
CMS13-350Q1000 generated
D01 generated
D02 generated
D02mu generated
E228-200-4Q5 generated
E228-200-5Q6 generated
E228-200-6Q7 generated
E228-200-7Q8 generated
E228-200-8Q9 generated
E228-300-11Q12 generated
E228-300-4Q5 generated
E228-300-5Q6 generated
E228-300-6Q7 generated
E228-300-7Q8 generated
E228-300-8Q9 generated
E228-400-11Q12 generated
E228-400-12Q13 generated
E228-400-13Q14 generated
E228-400-5Q6 generated
E228-400-6Q7 generated
E228-400-7Q8 generated
E228-400-8Q9 generated
LHCb7 generated
LH

In [11]:
display(negative_y_files)

['ATLAS8-116Q150',
 'ATLAS8-46Q66',
 'ATLAS13',
 'CMS7',
 'CMS8',
 'CMS13-106Q170',
 'CMS13-170Q350',
 'CMS13-350Q1000',
 'STAR510']

xf dependent

In [12]:
experiments =[
    "E605",
    "E772"
]
excludes = []

qT_if_integrate = True
Q_if_integrate = True
y_if_integrate = True

In [13]:
for experiment in experiments:

    exp_dir = before_dir/experiment
    file_names = [file.stem for file in sorted(exp_dir.glob("*.csv"))]

    save_dir = Path(after_dir) / "DY" / experiment
    save_dir.mkdir(parents=True, exist_ok=True)

    for file_name in (f for f in file_names if f not in excludes):

        path = exp_dir / f"{file_name}.csv"

        meta, df_raw = read_csv(path)

        def get_scalar(name):
            val = meta[name]
            if val.lower() == "true":
                return True
            elif val.lower() == "false":
                return False
            else:
                return val

        def get_array(name):
            return np.array(df_raw[name])

        # safety

        observable = observable_dict[experiment]
        if_normalized = get_scalar("Total cross-section nomalized")
        if if_normalized==True and observable!="1/σ*dσ/dqT":
            raise ValueError("observable type wrong")
        if if_normalized==False and observable=="1/σ*dσ/dqT":
            raise ValueError("observable type wrong")        

        # arrays

        sqrts_array = np.sqrt(get_array("s[GeV^2]"))
        isoscalarity = isoscalarity_dict[experiment]

        qT_min_array = get_array("qTMin[GeV]")
        qT_max_array = get_array("qTMax[GeV]")
        qT_mean_array = get_array("<qT>[GeV]")

        Q_min_array = get_array("Qmin[GeV]")
        Q_max_array = get_array("Qmax[GeV]")
        Q_mean_array = get_array("<Q>[GeV]")

        xf_min_array = get_array("yMin")
        xf_max_array = get_array("yMax")
        xf_mean_array = get_array("<y>")

        xsec_array = get_array("xSec")

        # fiducial cuts
        if_fiducial = get_array("FiducialCuts")
        pT_min_1 = get_array("kCut1[GeV]")
        pT_min_2 = get_array("kCut2[GeV]")
        eta_min = get_array("etaMin")
        eta_max = get_array("etaMax")

        # normalization
        norm_error = get_scalar("List of norm.errors (relative)")
        if_normalized = get_scalar("Total cross-section nomalized")

        # generate csv

        N = len(xsec_array)

        df = pd.DataFrame({

            "qT_if_integrate": [qT_if_integrate]*N,
            "qT_mean": qT_mean_array, "qT_min": qT_min_array, "qT_max": qT_max_array,

            "Q_if_integrate": [Q_if_integrate]*N,
            "Q_mean": Q_mean_array, "Q_min": Q_min_array, "Q_max": Q_max_array,

            "y_if_integrate": [y_if_integrate]*N,
            "xf_mean": xf_mean_array, "xf_min": xf_min_array, "xf_max": xf_max_array,

            "sqrts": sqrts_array,
            "isoscalarity": [isoscalarity]*N,
            "observable": [observable]*N,

            "if_fiducial": if_fiducial,
            "pT_min_1": pT_min_1,
            "pT_min_2": pT_min_2,
            "eta_min": eta_min,
            "eta_max": eta_max
        })

        # push errors

        def count_errors(label):
            pat = re.compile(rf"^{re.escape(label)}\d+$")
            return sum(1 for c in df_raw.columns if pat.match(c))
        count_errors('Uncorr.Err.')

        def get_errors(label, i):
            return get_array(f"{label}{i}")

        def push_errors(df, label, name):
            for i in range(count_errors(label)):
                df[f"{name}{i}"] = get_errors(label, i)
            return df   

        df["xsec"] = xsec_array

        df = push_errors(df, 'Uncorr.Err.', 'error_unc_')
        df = push_errors(df, 'Corr.Err.', 'error_add_')
        df["error_mult_0 "] = [norm_error]*N

        destination = after_dir / "DY" / experiment / f"{file_name}.csv"
        if destination.exists():
            raise FileExistsError(f"{file_name} already exists")
        df_rounded = df.round(6)
        df_rounded.to_csv(destination, index=False)

        print(f"{file_name} generated")

E605-10.5Q11.5 generated
E605-11.5Q13.5 generated
E605-13.5Q18 generated
E605-7Q8 generated
E605-8Q9 generated
E772-11Q12 generated
E772-12Q13 generated
E772-13Q14 generated
E772-14Q15 generated
E772-5Q6 generated
E772-6Q7 generated
E772-7Q8 generated
E772-8Q9 generated


Additional Treatments

Divide by qT binsize

In [14]:
experiments =[
    "LHCb_7",
    "LHCb_8",
]

In [15]:
for experiment in experiments:

    dir = Path(after_dir) / "DY" / experiment
    file_names = [file.stem for file in sorted(dir.glob("*.csv"))]

    for file_name in file_names:

        path = dir / f"{file_name}.csv"

        df = pd.read_csv(path)

        qT_bin_sizes = np.array(df["qT_max"] - df["qT_min"])
        df["xsec"] = df["xsec"] / qT_bin_sizes
        df["error_unc_0"] = df["error_unc_0"] / qT_bin_sizes
        df["error_unc_1"] = df["error_unc_1"] / qT_bin_sizes
        df["error_add_0"] = df["error_add_0"] / qT_bin_sizes
        df["error_add_1"] = df["error_add_1"] / qT_bin_sizes

        destination = after_dir / "DY" / experiment / f"{file_name}.csv"
        df_rounded = df.round(6)
        df_rounded.to_csv(destination, index=False)

        print(f"{file_name} generated")

LHCb7 generated
LHCb8 generated


pp change |y|

In [16]:
experiments =[
    'ATLAS_7',
    'ATLAS_8',
    'ATLAS_13',    
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    'PHENIX',
    'STAR'
]

In [17]:
for experiment in experiments:

    dir = Path(after_dir) / "DY" / experiment
    file_names = [file.stem for file in sorted(dir.glob("*.csv"))]

    for file_name in file_names:
        if file_name in negative_y_files:
            
            path = dir / f"{file_name}.csv"

            df = pd.read_csv(path)

            N = len(df["xsec"])
            df["y_min"] = [0]*N

            if df["isoscalarity"][0] != 1:
                raise ValueError("isoscalarity not 1")

            destination = after_dir / "DY" / experiment / f"{file_name}.csv"
            df_rounded = df.round(6)
            df_rounded.to_csv(destination, index=False)

            print(f"{file_name} generated")

ATLAS8-116Q150 generated
ATLAS8-46Q66 generated
ATLAS13 generated
CMS7 generated
CMS8 generated
CMS13-106Q170 generated
CMS13-170Q350 generated
CMS13-350Q1000 generated
STAR510 generated


In [18]:
for experiment in experiments:

    dir = Path(after_dir) / "DY" / experiment
    file_names = [file.stem for file in sorted(dir.glob("*.csv"))]

    for file_name in file_names:
            
        path = dir / f"{file_name}.csv"

        df = pd.read_csv(path)

        N = len(df["xsec"])
        df["y_min"] = [0]*N

        if df["isoscalarity"][0] == 1 and df["y_min"][0] < 0:
            raise ValueError("negative ymin for pp")

inclusive

In [19]:
experiments =[
    "CDF_I",
    "CDF_II",
    "D0_I",
    "D0_II",
    "D0_II_mu",
]

for experiment in experiments:

    dir = Path(after_dir) / "DY" / experiment
    file_names = [file.stem for file in sorted(dir.glob("*.csv"))]

    for file_name in file_names:

        path = dir / f"{file_name}.csv"

        df = pd.read_csv(path)

        qT_bin_sizes = np.array(df["qT_max"] - df["qT_min"])
        df["y_min"] = -3.5
        df["y_max"] = 3.5

        destination = after_dir / "DY" / experiment / f"{file_name}.csv"
        df_rounded = df.round(6)
        df_rounded.to_csv(destination, index=False)

        print(f"{file_name} generated")

CDF1 generated
CDF2 generated
D01 generated
D02 generated
D02mu generated


Divide by 1000

In [20]:
experiments =[
    "E288",
]

In [21]:
for experiment in experiments:

    dir = Path(after_dir) / "DY" / experiment
    file_names = [file.stem for file in sorted(dir.glob("*.csv"))]
    
    for file_name in file_names:   

        path = dir / f"{file_name}.csv"

        df = pd.read_csv(path)

        qT_bin_sizes = np.array(df["qT_max"] - df["qT_min"])
        df["xsec"] = df["xsec"] / 1000
        df["error_unc_0"] = df["error_unc_0"] / 1000

        destination = after_dir / "DY" / experiment / f"{file_name}.csv"
        df_rounded = df.round(6)
        df_rounded.to_csv(destination, index=False)

        print(f"{file_name} generated")

E228-200-4Q5 generated
E228-200-5Q6 generated
E228-200-6Q7 generated
E228-200-7Q8 generated
E228-200-8Q9 generated
E228-300-11Q12 generated
E228-300-4Q5 generated
E228-300-5Q6 generated
E228-300-6Q7 generated
E228-300-7Q8 generated
E228-300-8Q9 generated
E228-400-11Q12 generated
E228-400-12Q13 generated
E228-400-13Q14 generated
E228-400-5Q6 generated
E228-400-6Q7 generated
E228-400-7Q8 generated
E228-400-8Q9 generated
